# Скоринг ИП — MVP с БКИ и подтверждением сотрудника

Данные синтетические, ответы ФНС/БКИ имитируются словарями. Все веса, пороги, лимиты и правила R1–R12 — **учебные проектные допущения**, не кредитная политика Альфа-Банка.

**Поток:** проверка анкеты → проверка согласия → получение результата БКИ → скоринг → предварительное решение → подтверждение/разбор сотрудником. Реальная выдача кредита не реализуется.

Запускайте ячейки сверху вниз. В ячейке с `my_ip` можно менять исходные данные и ответы заглушек.

## 1. Балльная модель — без изменения весов

| Фактор | Баллы |
|---|---|
| Возраст ИП | ≥24 мес.: 20; 6–23: 10; <6: 0 |
| Изменение выручки | >10%: 25; от −10% до +10%: 15; меньше −10%: 0 |
| Долговая нагрузка | ≤30%: 30; >30–50%: 15; >50%: 0 |
| Кредитная история из БКИ | Чистая: 25; небольшие погашенные просрочки: 10; серьёзные: 0 |

Шкала риска: 75–100 низкий, 50–74 средний, 0–49 высокий. Значение `credit_history` в анкете не принимается: история поступает **только** из `bki_result`.

В реальном продукте кредитная история может требовать сложной интерпретации. Здесь используется упрощённая синтетическая категория.

In [ ]:
# Исходные константы не меняются.
MIN_AMOUNT = 100_000
MAX_AMOUNT = 3_000_000
HISTORY_POINTS = {"clean": 25, "minor_closed": 10, "serious": 0}


def risk_group(score):
    if not 0 <= score <= 100:
        raise ValueError("Балл должен быть от 0 до 100")
    if score >= 75:
        return "Низкий"
    if score >= 50:
        return "Средний"
    return "Высокий"


def calculate_score(ip):
    """Расчёт прежних четырёх факторов только после успешной проверки БКИ."""
    age = ip["age_months"]
    revenue = ip["revenue"]
    previous = ip["previous_revenue"]
    debt = ip["debt_payments"]
    bki = ip["bki_result"]
    if bki.get("status") != "SUCCESS":
        raise ValueError("Нет успешного результата БКИ")
    history = bki.get("credit_history")
    if any(type(x) not in (int, float) for x in (age, revenue, previous, debt)):
        raise ValueError("Некорректный тип финансовых данных")
    if age < 0 or revenue <= 0 or previous <= 0 or debt < 0:
        raise ValueError("Некорректные финансовые данные")
    if history not in HISTORY_POINTS:
        raise ValueError("Неизвестный тип кредитной истории из БКИ")

    growth = (revenue - previous) / previous * 100
    burden = debt / revenue * 100
    points = {
        "Возраст": 20 if age >= 24 else (10 if age >= 6 else 0),
        "Выручка": 25 if growth > 10 else (15 if growth >= -10 else 0),
        "Долговая нагрузка": 30 if burden <= 30 else (15 if burden <= 50 else 0),
        "Кредитная история": HISTORY_POINTS[history],
    }
    score = sum(points.values())
    return {
        "score": score,
        "risk": risk_group(score),
        "points": points,
        "growth_percent": round(growth, 2),
        "burden_percent": round(burden, 2),
    }

print("Балльная модель 20/25/30/25 загружена")

## 2. Маршрутизация R1–R11 и финальное решение R12

- **R1:** проверка полей заявки **до** запроса БКИ.
- **R3:** нет обязательных данных или согласия БКИ; запрос БКИ не выполняется.
- **R4:** заглушка БКИ/ФНС не ответила (`WAIT_EXTERNAL`); не приравниваем отсутствие истории к плохой истории.
- **R2:** проверенный стоп-фактор: неактивный ИП или текущая просрочка ≥30 дней.
- **R5:** противоречивые или недопустимые данные.
- **R6–R7:** высокий риск или лимит ниже минимального — **рекомендация отказа**, не подтверждённое решение сотрудника.
- **R8–R9:** низкий риск — предварительное одобрение (`PRE_APPROVED`), заявка ожидает сотрудника (`WAITING_EMPLOYEE_APPROVAL`).
- **R10–R11:** средний риск — `MANUAL_REVIEW`.
- **R12:** отдельное подтверждение/отказ сотрудника, **не** внутри скоринга.

Лимиты сохранены: при низком риске `min(3 млн, 3 × месячная выручка)`, при среднем — `min(1 млн, месячная выручка)`. Вопрос об автоматическом финальном отказе при стоп-факторе команда должна согласовать отдельно; здесь скоринг только рекомендует отказ.

In [ ]:
def validate_request(ip):
    """Pre-BKI проверка: её следует вызывать прежде любого внешнего запроса."""
    amount, term, purpose = (ip.get(k) for k in ("requested_amount", "term_months", "purpose"))
    if (type(amount) not in (int, float) or not MIN_AMOUNT <= amount <= MAX_AMOUNT
            or type(term) is not int or not 6 <= term <= 36
            or purpose != "working_capital"):
        return "R1", "CORRECTION_REQUIRED", "Исправьте сумму, срок или цель"
    fields = ("age_months", "revenue", "previous_revenue", "debt_payments", "active")
    if any(ip.get(k) is None for k in fields) or ip.get("bki_consent") is not True:
        return "R3", "NEED_INFO", "Нет обязательных данных или согласия БКИ"
    return None


def evaluate(ip):
    """Рекомендация скоринга; никогда не выдаёт финальное APPROVED."""
    answer = {"score": None, "risk": None, "limit": None,
              "offered_amount": None, "rule": None, "decision": None,
              "application_status": None, "requires_employee_confirmation": False,
              "points": None, "reason": ""}

    def result(rule, decision, reason, status=None, amount=None):
        answer.update(rule=rule, decision=decision, reason=reason,
                      application_status=status or decision, offered_amount=amount,
                      requires_employee_confirmation=(decision == "PRE_APPROVED"))
        return answer

    invalid = validate_request(ip)
    if invalid:
        return result(*invalid)

    # В реальном backend этот блок означает вызов заранее завершённой заглушки БКИ.
    # Модель не инициирует запрос и не списывает попытку проверки.
    bki = ip.get("bki_result")
    if not isinstance(bki, dict) or bki.get("status") in (None, "PENDING", "ERROR"):
        return result("R4", "WAIT_EXTERNAL", "Ожидание успешного результата БКИ")
    if bki.get("status") != "SUCCESS":
        return result("R5", "MANUAL_REVIEW", "Некорректный статус ответа БКИ")
    if ip.get("external_available") is False:
        return result("R4", "WAIT_EXTERNAL", "Другая обязательная проверка недоступна")

    overdue = bki.get("overdue_days")
    if (type(overdue) is int and overdue >= 30) or ip.get("active") is False:
        try:
            diagnostic = calculate_score(ip)
            answer.update(score=diagnostic["score"], risk=diagnostic["risk"],
                          points=diagnostic["points"])
        except (KeyError, TypeError, ValueError, ZeroDivisionError):
            pass
        return result("R2", "REJECT_RECOMMENDED", "Стоп-фактор: ИП неактивен или просрочка от 30 дней")
    if type(overdue) is not int or overdue < 0:
        return result("R5", "MANUAL_REVIEW", "Некорректная текущая просрочка в БКИ")

    try:
        data = calculate_score(ip)
    except (KeyError, TypeError, ValueError, ZeroDivisionError):
        return result("R5", "MANUAL_REVIEW", "Противоречивые или отсутствующие данные")
    answer.update(score=data["score"], risk=data["risk"], points=data["points"])

    if data["risk"] == "Высокий":
        return result("R6", "REJECT_RECOMMENDED", "Высокий риск")

    revenue = ip["revenue"]
    limit = (min(3_000_000, 3 * revenue) if data["risk"] == "Низкий"
             else min(1_000_000, revenue))
    answer["limit"] = int(limit)
    if limit < MIN_AMOUNT:
        return result("R7", "REJECT_RECOMMENDED", "Расчётный лимит ниже 100 тыс. ₽")

    amount = ip["requested_amount"]
    if data["risk"] == "Низкий":
        if amount <= limit:
            return result("R8", "PRE_APPROVED", "Сумма предварительно доступна",
                          "WAITING_EMPLOYEE_APPROVAL", int(amount))
        return result("R9", "PRE_APPROVED", "Предварительно предложена меньшая сумма",
                      "WAITING_EMPLOYEE_APPROVAL", int(limit))
    if amount <= limit:
        return result("R10", "MANUAL_REVIEW", "Средний риск — необходимо решение сотрудника")
    return result("R11", "MANUAL_REVIEW", "Средний риск, заявка выше лимита")


def employee_decision(ip, recommendation, action, approved_amount=None, comment=""):
    """Упрощённый R12 для демонстрации. В реальном сервисе роль, аудит и блокировки — в backend."""
    if recommendation["decision"] not in ("PRE_APPROVED", "MANUAL_REVIEW"):
        raise ValueError("Заявка не готова к решению сотрудника")
    if recommendation["rule"] == "R5":
        raise ValueError("Сначала необходимо уточнить противоречивые данные")
    if action not in ("approve", "reject") or not comment.strip():
        raise ValueError("Укажите approve/reject и комментарий")
    if action == "reject":
        return {"rule": "R12", "decision": "REJECTED", "approved_amount": None, "comment": comment}
    limit = recommendation["limit"]
    maximum = min(ip["requested_amount"], limit) if limit is not None else None
    if (type(approved_amount) is not int or maximum is None
            or not MIN_AMOUNT <= approved_amount <= maximum):
        raise ValueError("Сумма одобрения превышает запрошенную сумму или расчётный лимит")
    return {"rule": "R12", "decision": "APPROVED", "approved_amount": approved_amount,
            "comment": comment}

print("Маршрутизация R1–R11 и отдельное решение R12 загружены")

## 3. Пример одного предпринимателя

`bki_result` здесь — **синтетический готовый ответ** БКИ. В настоящей системе его получает backend после проверки заявки и согласия клиента. Не используйте реальный доступ к БКИ из Jupyter Notebook.

In [ ]:
my_ip = {
    "age_months": 36,
    "revenue": 500_000,
    "previous_revenue": 450_000,
    "debt_payments": 100_000,
    "active": True,
    "bki_consent": True,
    "external_available": True,
    "bki_result": {"status": "SUCCESS", "credit_history": "clean", "overdue_days": 0},
    "requested_amount": 1_000_000,
    "term_months": 12,
    "purpose": "working_capital",
}

answer = evaluate(my_ip)
for key, value in answer.items():
    print(f"{key}: {value}")

## 4. Контрольные профили T1–T28

Профили синтетические, без базы данных. Тесты T1–T20 сохранены по смыслу, их ожидаемые **финальные результаты** обновлены на предварительные рекомендации. Добавлены случаи ожидания ответа БКИ, запрета обращения при ошибке, проверки R12 и лимитов.

In [ ]:
base_ip = {
    "age_months": 48, "revenue": 800_000, "previous_revenue": 680_000,
    "debt_payments": 160_000, "active": True, "bki_consent": True,
    "external_available": True, "requested_amount": 2_000_000,
    "term_months": 12, "purpose": "working_capital",
    "bki_result": {"status": "SUCCESS", "credit_history": "clean", "overdue_days": 0},
}


def make_ip(**changes):
    ip = {**base_ip, "bki_result": dict(base_ip["bki_result"])}
    if "bki_result" in changes:
        ip["bki_result"] = changes.pop("bki_result")
    return {**ip, **changes}


def with_bki(**bki_changes):
    return {**base_ip["bki_result"], **bki_changes}


tests = [
    ("T1", "Сильный, сумма в лимите", make_ip(), 100, "R8", "PRE_APPROVED", 2_000_000),
    ("T2", "Сильный, сумма больше лимита", make_ip(requested_amount=3_000_000), 100, "R9", "PRE_APPROVED", 2_400_000),
    ("T3", "Средний, сумма в лимите", make_ip(age_months=18, revenue=600_000, previous_revenue=600_000, debt_payments=240_000, requested_amount=500_000), 65, "R10", "MANUAL_REVIEW", None),
    ("T4", "Средний, запрос выше лимита", make_ip(age_months=18, revenue=600_000, previous_revenue=600_000, debt_payments=240_000, requested_amount=900_000), 65, "R11", "MANUAL_REVIEW", None),
    ("T5", "Высокий риск", make_ip(age_months=4, revenue=300_000, previous_revenue=375_000, debt_payments=180_000, bki_result=with_bki(credit_history="minor_closed"), requested_amount=200_000), 10, "R6", "REJECT_RECOMMENDED", None),
    ("T6", "Текущая просрочка 45 дней", make_ip(bki_result=with_bki(overdue_days=45, credit_history="serious")), 75, "R2", "REJECT_RECOMMENDED", None),
    ("T7", "Выручка неизвестна", make_ip(revenue=None), None, "R3", "NEED_INFO", None),
    ("T8", "Ровно 75 баллов", make_ip(revenue=400_000, previous_revenue=400_000, debt_payments=160_000, requested_amount=1_000_000), 75, "R8", "PRE_APPROVED", 1_000_000),
    ("T9", "Ровно 50 баллов", make_ip(age_months=12, revenue=300_000, previous_revenue=300_000, debt_payments=120_000, bki_result=with_bki(credit_history="minor_closed"), requested_amount=250_000), 50, "R10", "MANUAL_REVIEW", None),
    ("T10", "Ближайший балл ниже 50: 45", make_ip(revenue=300_000, previous_revenue=300_000, debt_payments=180_000, bki_result=with_bki(credit_history="minor_closed"), requested_amount=200_000), 45, "R6", "REJECT_RECOMMENDED", None),
    ("T11", "Ближайший балл ниже 75: 70", make_ip(debt_payments=320_000, bki_result=with_bki(credit_history="minor_closed"), requested_amount=500_000), 70, "R10", "MANUAL_REVIEW", None),
    ("T12", "Неактивный ИП", make_ip(active=False), 100, "R2", "REJECT_RECOMMENDED", None),
    ("T13", "Срок 48 месяцев", make_ip(term_months=48), None, "R1", "CORRECTION_REQUIRED", None),
    ("T14", "Предыдущая выручка нулевая", make_ip(previous_revenue=0), None, "R5", "MANUAL_REVIEW", None),
    ("T15", "БКИ не ответило", make_ip(bki_result={"status": "ERROR"}), None, "R4", "WAIT_EXTERNAL", None),
    ("T16", "Нет согласия на отчёт БКИ", make_ip(bki_consent=False, bki_result=None), None, "R3", "NEED_INFO", None),
    ("T17", "Лимит ниже минимума", make_ip(revenue=20_000, previous_revenue=20_000, debt_payments=8_000, requested_amount=100_000), 75, "R7", "REJECT_RECOMMENDED", None),
    ("T18", "Сумма точно на лимите", make_ip(requested_amount=2_400_000), 100, "R8", "PRE_APPROVED", 2_400_000),
    ("T19", "Средний риск до решения сотрудника", make_ip(age_months=18, revenue=600_000, previous_revenue=600_000, debt_payments=240_000, requested_amount=900_000), 65, "R11", "MANUAL_REVIEW", None),
    ("T20", "Противоречивый отрицательный долг", make_ip(debt_payments=-100), None, "R5", "MANUAL_REVIEW", None),
    ("T21", "Ошибка суммы и нет БКИ", make_ip(requested_amount=-1, bki_result=None), None, "R1", "CORRECTION_REQUIRED", None),
    ("T22", "Нет согласия и нет БКИ", make_ip(bki_consent=False, bki_result=None), None, "R3", "NEED_INFO", None),
    ("T23", "Ожидание БКИ", make_ip(bki_result={"status": "PENDING"}), None, "R4", "WAIT_EXTERNAL", None),
    ("T24", "Низкий риск 100 баллов", make_ip(), 100, "R8", "PRE_APPROVED", 2_000_000),
    ("T25", "Средний риск 65 баллов", make_ip(age_months=18, revenue=600_000, previous_revenue=600_000, debt_payments=240_000, requested_amount=500_000), 65, "R10", "MANUAL_REVIEW", None),
    ("T26", "Высокий риск 45 баллов", make_ip(revenue=300_000, previous_revenue=300_000, debt_payments=180_000, bki_result=with_bki(credit_history="minor_closed"), requested_amount=200_000), 45, "R6", "REJECT_RECOMMENDED", None),
    ("T27", "Сильный балл, но стоп-фактор", make_ip(bki_result=with_bki(overdue_days=35)), 100, "R2", "REJECT_RECOMMENDED", None),
    ("T28", "Предварительное одобрение ещё не финальное", make_ip(), 100, "R8", "PRE_APPROVED", 2_000_000),
]
print("Количество тестовых профилей:", len(tests))

## 5. Автоматические проверки

`assert` проверяет баллы, правило, рекомендацию и сумму для всех 28 профилей. Отдельные тесты проверяют, что финальный статус `APPROVED` появляется только после решения сотрудника и что отсутствие ответа БКИ не превращается в плохую историю.

In [ ]:
print(f"{'ID':<5} {'Балл':<8} {'Правило':<8} {'Рекомендация':<22} {'Сумма':<12} Статус")
print("-" * 85)
for name, description, ip, expected_score, expected_rule, expected_decision, expected_amount in tests:
    actual = evaluate(ip)
    assert actual["score"] == expected_score, f"{name}: балл {actual['score']}"
    assert actual["rule"] == expected_rule, f"{name}: правило {actual['rule']}"
    assert actual["decision"] == expected_decision, f"{name}: решение {actual['decision']}"
    assert actual["offered_amount"] == expected_amount, f"{name}: сумма {actual['offered_amount']}"
    assert actual["decision"] != "APPROVED", f"{name}: недопустимое финальное одобрение"
    print(f"{name:<5} {str(actual['score']):<8} {actual['rule']:<8} {actual['decision']:<22} {str(actual['offered_amount']):<12} OK")

assert evaluate(make_ip(requested_amount=2_400_001))["rule"] == "R9"
for score, expected in [(49, "Высокий"), (50, "Средний"), (74, "Средний"), (75, "Низкий")]:
    assert risk_group(score) == expected

low = make_ip()
preliminary = evaluate(low)
assert preliminary["application_status"] == "WAITING_EMPLOYEE_APPROVAL"
assert preliminary["requires_employee_confirmation"] is True
final = employee_decision(low, preliminary, "approve", 2_000_000, "Проверки подтверждены")
assert final["decision"] == "APPROVED" and final["rule"] == "R12"

medium = make_ip(age_months=18, revenue=600_000, previous_revenue=600_000,
                 debt_payments=240_000, requested_amount=900_000)
assert employee_decision(medium, evaluate(medium), "approve", 600_000,
                         "Дополнительная проверка выполнена")["decision"] == "APPROVED"
try:
    employee_decision(medium, evaluate(medium), "approve", 700_000, "Превышен лимит")
except ValueError:
    pass
else:
    raise AssertionError("Нельзя одобрить сумму выше лимита")

for invalid_ip in (make_ip(bki_result=None), make_ip(bki_consent=False),
                   make_ip(bki_result=with_bki(overdue_days=45))):
    candidate = evaluate(invalid_ip)
    try:
        employee_decision(invalid_ip, candidate, "approve", 100_000, "Тест")
    except ValueError:
        pass
    else:
        raise AssertionError("Недопустимое одобрение")

print("\nВсе 28 сценариев, проверки границ и R12 пройдены")

## 6. Несколько ИП без базы данных

Получаем распределение предварительных рекомендаций, а не статистику реально выданных кредитов.

In [ ]:
entrepreneurs = [ip for _, _, ip, *_ in tests]
decisions_count = {}
for ip in entrepreneurs:
    decision = evaluate(ip)["decision"]
    decisions_count[decision] = decisions_count.get(decision, 0) + 1
print("Распределение рекомендаций по 28 синтетическим профилям:")
for decision, count in sorted(decisions_count.items()):
    print(f"{decision}: {count}")

## 7. Интеграция с backend

Backend валидирует форму и фиксирует согласие на БКИ **до обращения во внешнюю заглушку**. Затем передаёт в `evaluate(ip)` заполненные поля и ответ БКИ в виде `bki_result`. При `PRE_APPROVED` заявка переходит в `WAITING_EMPLOYEE_APPROVAL`; при `MANUAL_REVIEW` — в очередь специалиста. `REJECT_RECOMMENDED` означает лишь рекомендацию модели; финальный маршрут согласуйте с аналитиком 3.

Действие сотрудника (`APPROVED` / `REJECTED`) принимает **отдельный backend endpoint**, который проверяет полномочия, версии, переходы, лимиты и пишет аудит. Функция `employee_decision` здесь демонстрирует бизнес-ограничения R12, но не заменяет защищённый сервер.

**Нет** прямых запросов в БКИ, SQLite, обучения ML или выдачи денег. Формулы и пороги сохранены.